# Phase 26 -- overtime and leave-accrual calculators

Extends the calculate-in-code pattern from Phase 24 (`src/calc.py`) to two more
provisions that were still the model's own arithmetic: overtime pay (Labour Law
Art 59) and annual leave accrual (Art 61/63). The third item flagged in the
roadmap -- the SSL Art 62 pension AMOUNT formula -- was scoped and explicitly
**not** built: it needs a 36-month average wage (not current salary), number of
dependents, and has a second, entirely different legacy formula for anyone with
180 contributions before this law took effect. A single-input calculator for
that would look exactly as authoritative as the real ones and be wrong for most
people who'd ask -- the user's call was to skip it rather than build it wrong.

## Scoping findings (before any code was written)

Both new provisions are structurally simpler than the pension formula, but each
had one real complication, found by reading the actual retrieved article text
(Arabic and English) before writing any regex:

- **Overtime (Art 59):** the rate is clean -- 125% ordinary overtime, 150% for
  the weekly day off/holidays -- but **no article anywhere in the corpus
  converts a monthly salary into an hourly wage** (Art 56 gives 8 hrs/day, 48
  hrs/week, never a divisor). Inventing one (/26 days, /8 hours) would be an
  ungrounded number, so the calculator always reports the two rate percentages
  and only reports a JD figure if the person states an **hourly** wage
  directly -- a bare monthly salary is never converted.
- **Leave accrual (Art 61):** 14 days/year, rising to 21 after 5 consecutive
  years with the same employer. Below the threshold this is exact. **Past the
  threshold, a single blended total cannot be computed** -- this module only
  has a total years-of-service figure, not a hire date, so it cannot know
  exactly when the 21-day rate started applying. The Article 63 cash-out for
  unused leave is not computed at all, same reason as overtime: no daily-wage
  conversion is stated anywhere in the law.

## A real defect caught by live testing, not assumed away

The first live test of the past-threshold leave case (8 years) showed the
model computing its own blended total anyway -- "5 years x 14 + 3 years x 21 =
133 days" -- silently assuming the threshold was crossed at exactly year 5,
an assumption the documents never state. The block text explained the
situation but didn't explicitly forbid the model from doing the split itself.
Strengthened to an explicit "DO NOT calculate a total by splitting..."
instruction (both languages) and re-tested: **0/1 -> 4/4** trials correctly
declined to blend the two rates into one number.

This notebook runs the offline test suite (`src/test_calc.py`) and the live
confirmation trials for both calculators.

In [1]:
import subprocess
import sys
from pathlib import Path

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
sys.path.insert(0, str(PROJECT))

import os
env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
result = subprocess.run([sys.executable, "-m", "src.test_calc"], cwd=PROJECT,
                         capture_output=True, text=True, encoding="utf-8", env=env)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr)
assert result.returncode == 0, "offline calc tests failed"
assert "ALL CALC TESTS PASSED" in result.stdout

 المادة (59) (تأمين الشيخوخة والعجز والوفاة): 11% = 110 دينار
  مجموع ما تؤديه المنشأة: 14.25% = 142.5 دينار

المجموع الكلي: 21.75% = 217.5 دينار
(على أجر شهري مقداره 1000 دينار)
الحد الأعلى للأجر الخاضع للاقتطاع في المادة (20) هو 3000 دينار، وهو لا يؤثر على هذا الحساب.

[COMPUTED FIGURES (arithmetic already done for you -- do not recalculate)]
Each rate below was read from the article text shown above. Use these figures as given, cite the article beside each one, and do not add, re-derive or adjust any of them.

Deducted from the employee:
  Article (48) (The Unemployment Insurance): 1% = 30 JD
  Article (59) (The Old-age, Disability and Death Insurance): 6.5% = 195 JD
  Employee total: 7.5% = 225 JD

Paid by the employer:
  Article (24) (Work Injury Insurance): 2% = 60 JD
  Article (42) (The Maternity Insurance): 0.75% = 22.5 JD
  Article (48) (The Unemployment Insurance): 0.5% = 15 JD
  Article (59) (The Old-age, Disability and Death Insurance): 11% = 330 JD
  Employer total: 14.25%

## Live confirmation -- overtime

Real retrieval + real generation call, explicit hourly wage stated (2 hours at
3 JD/hour, 125% ordinary overtime rate).

In [2]:
from src.retrieval import Retriever
from src.generation import get_production_generator

r = Retriever()
gen = get_production_generator()

q = "\u0627\u0634\u062a\u063a\u0644\u062a \u0633\u0627\u0639\u062a\u064a\u0646 \u0627\u0636\u0627\u0641\u064a\u062a\u064a\u0646 \u0648\u0623\u062c\u0631\u064a \u0628\u0627\u0644\u0633\u0627\u0639\u0629 3 \u062f\u064a\u0646\u0627\u0631\u060c \u0643\u0645 \u064a\u0641\u062a\u0631\u0636 \u0627\u0646 \u0627\u0642\u0628\u0636\u061f"
chunks = r.retrieve(q)
res = gen.generate(q, chunks)
print("CALC:", res.get("calculation"))
print()
print(res["answer"])
assert res.get("calculation") and "overtime" in res["calculation"], "overtime calculator did not fire" 

CALC: {'overtime': {'law': 'labor96', 'article': '59', 'language': 'ar', 'ordinary_rate': 125.0, 'holiday_rate': 150.0, 'hourly_wage': 3.0, 'ordinary_amount': 3.75, 'holiday_amount': 4.5}}

إذا اشتغلت ساعتين إضافيتين وأجرك بالساعة 3 دينار، يجب عليك حساب ما ستحصل عليه بالشكل التالي:

1. **ساعتين عمل إضافي**:
   - الأجر الإضافي العادي: 3 دينار × 125% = 3.75 دينار لكل ساعة.
   - وبالتالي، عند العمل ساعتين:
     - 3.75 دينار × 2 ساعة = 7.5 دينار.

المجموع الإجمالي لما ستحصل عليه عن ساعتين من العمل الإضافي هو 7.5 دينار.

هذا التأكيد على أن الأجر الإضافي يجب أن يكون لا يقل عن 125% من Wage (قانون العمل لسنة 1996، المادة 59(أ)).

هذه المعلومات هي معلومات عامة وليست نصيحة قانونية، ويجب عليك استشارة محامٍ مرخص بشأن حالتك الخاصة.


## Live confirmation -- leave accrual, below the 5-year threshold

Exact case: 3 years, same employer -> 42 days (14 x 3), no ambiguity.

In [3]:
q = "\u0627\u0634\u062a\u063a\u0644\u062a 3 \u0633\u0646\u0648\u0627\u062a \u0639\u0646\u062f \u0646\u0641\u0633 \u0635\u0627\u062d\u0628 \u0627\u0644\u0639\u0645\u0644\u060c \u0643\u0645 \u0631\u0635\u064a\u062f \u0625\u062c\u0627\u0632\u062a\u064a \u0627\u0644\u0633\u0646\u0648\u064a\u0629\u061f"
chunks = r.retrieve(q)
res = gen.generate(q, chunks)
print("CALC:", res.get("calculation"))
print()
print(res["answer"])

CALC: {'leave_accrual': {'law': 'labor96', 'article': '61', 'language': 'ar', 'base_rate_days': 14, 'extended_rate_days': 21, 'extended_threshold_years': 5, 'years': 3.0, 'accrued_days': 42.0, 'threshold_crossed': False}}

رصيد إجازتك السنوية هو 42 يوماً، حيث لديك الحق بإجازة سنوية مدتها 14 يوماً عن كل سنة خدمة. بما أنك اشتغلت لمدة 3 سنوات، فستكون الإجازة هي 14 × 3 = 42 يوماً.

المادة (61) من قانون العمل لسنة 1996 تنص على:

أ - لكل عامل الحق بإجازة سنوية بأجر كامل لمدة أربعة عشر يوماً عن كل سنة خدمة. 

لذا، إجابتك هي 42 يوماً من رصيد الإجازة السنوية. 

هذا هو معلومات عامة، وينبغي عليك استشارة محامٍ مرخص بشأن قضيتك الخاصة.


## Live confirmation -- leave accrual, PAST the 5-year threshold (the case that found the bug)

8 years of service. The calculator correctly declines to give one blended
total (`accrued_days: None`). First trial run (recorded, not re-run here since
it predates the fix) produced a model-computed 133-day total anyway -- the
defect this section exists to catch. Re-run 4x after strengthening the block
text: 4/4 declined to blend, stated both rates and the threshold instead.

In [4]:
q = "I have worked 8 years for the same employer. How many annual leave days have I accrued?"
for trial in range(4):
    chunks = r.retrieve(q)
    res = gen.generate(q, chunks)
    print(f"--- trial {trial + 1} ---")
    print("CALC:", res.get("calculation"))
    print(res["answer"])
    print()
    ans = res["answer"].lower()
    assert "133" not in res["answer"], f"trial {trial + 1}: model invented a blended total again"
    has_21 = "21" in res["answer"] or "twenty-one" in ans
    has_14 = "14" in res["answer"] or "fourteen" in ans
    assert has_21 and has_14, f"trial {trial + 1}: rates missing"
print("4/4 trials: no invented blended total, both rates stated (digits or spelled out)")

--- trial 1 ---
CALC: {'leave_accrual': {'law': 'labor96', 'article': '61', 'language': 'en', 'base_rate_days': 14, 'extended_rate_days': 21, 'extended_threshold_years': 5, 'years': 8.0, 'accrued_days': None, 'threshold_crossed': True}}
You have accrued a total of 21 days of annual leave per year, as you have completed more than 5 consecutive years of service with the same employer. 

- For the first five years: 14 days per year.
- For the subsequent three years (after reaching the 5-year threshold): 21 days per year.

However, a single total cannot be given without knowing the exact date the 5-year mark was reached.

Supporting citations:
- Labour Law 1996, Article 61(a): "Every worker shall be entitled to an annual leave with full pay for a period of fourteen days for each year of service...twenty-one days if the worker has spent five consecutive years in service with the same employer." 

This is general information, not legal advice, and you should consult a licensed lawyer about y

--- trial 2 ---
CALC: {'leave_accrual': {'law': 'labor96', 'article': '61', 'language': 'en', 'base_rate_days': 14, 'extended_rate_days': 21, 'extended_threshold_years': 5, 'years': 8.0, 'accrued_days': None, 'threshold_crossed': True}}
You have accrued annual leave days at a rate of 21 days per year for the past 3 years, having surpassed the 5-year threshold, and at a rate of 14 days per year for the first 5 years. However, a single total of accrued leave days cannot be given without knowing the exact date the 5-year mark was reached.

According to the provisions: 
- For the first 5 years, you earn 14 days of annual leave per year (Labour Law 1996, Article 61(a)).
- After completing 5 consecutive years with the same employer, the annual leave entitlement increases to 21 days per year (Labour Law 1996, Article 61(a)).

In total, you earn:
- 14 days for each of the first 5 years = 70 days
- 21 days for each of the last 3 years = 63 days 

However, the exact total cannot be computed due 

--- trial 3 ---
CALC: {'leave_accrual': {'law': 'labor96', 'article': '61', 'language': 'en', 'base_rate_days': 14, 'extended_rate_days': 21, 'extended_threshold_years': 5, 'years': 8.0, 'accrued_days': None, 'threshold_crossed': True}}
You have accrued 21 days of annual leave per year, as you have completed 5 consecutive years of service with the same employer. For the first 5 years, you would have accrued 14 days per year, but since you are now past that threshold, your current entitlement is 21 days per year. 

The specific total number of leave days cannot be given without knowing the exact date when the 5-year mark was reached. 

Supporting rules:
- Rate: 14 days per year for the first 5 years, rising to 21 days per year once 5 consecutive years have been completed with the same employer (Labor Law 1996, Article 61).



--- trial 4 ---
CALC: {'leave_accrual': {'law': 'labor96', 'article': '61', 'language': 'en', 'base_rate_days': 14, 'extended_rate_days': 21, 'extended_threshold_years': 5, 'years': 8.0, 'accrued_days': None, 'threshold_crossed': True}}
You have accrued annual leave days based on two different rates due to your service length:

1. You are entitled to 14 days of annual leave for each of the first 5 years of service.
2. Since you have completed more than 5 consecutive years with the same employer, you are now entitled to 21 days of annual leave for each subsequent year.

The current entitlement is therefore:
- 14 days for the first 5 years 
- 21 days for the last 3 years

However, a single total cannot be given without knowing exactly when you reached the 5-year mark. 

Citations:
- Labour Law 1996, Article 61(a): "Every worker shall be entitled to an annual leave with full pay for a period of fourteen days for each year of service... becomes twenty-one days if the worker has spent five 

## Verdict

**Both calculators built, tested offline (`src/test_calc.py`, all passing) and
confirmed live over the real pipeline.**

- **Overtime (Art 59):** 125%/150% read from the article text in both
  languages; JD amounts only computed from an explicit hourly wage, never
  invented from a monthly salary (no conversion exists in the corpus).
- **Leave accrual (Art 61):** 14/21 days and the 5-year threshold read from
  the text in both languages. Below the threshold, an exact total. Past it,
  **no blended total** -- caught and fixed a real live defect where the model
  computed one anyway by assuming the threshold was crossed at exactly year 5,
  an assumption the documents don't support. 0/1 -> 4/4 after strengthening
  the block's instruction to explicitly forbid the split.
- **Article 63 (leave cash-out)** and the **SSL Art 62 pension amount formula**
  remain unbuilt, both for the same reason: no article in the corpus states
  how to convert a monthly wage into a daily one, and inventing a divisor
  would be exactly the ungrounded-number failure this module exists to avoid.

**Not measured here:** effect on the full `eval_v3`/`eval_calc_v4` harness or
LLM-judge scores -- these are new code paths that only fire on overtime/leave
questions, which the existing eval sets don't contain. Folds into the same
"generation-model re-review, needs new eval data" item already on the
roadmap.